In [ ]:
from google.colab import drive

# Mount Google Drive so data files persist across Colab sessions.

drive.mount("/content/gdrive")
print("Drive mounted")

In [ ]:
from pathlib import Path

# Base project directory on Drive. All data files are stored under DATA_DIR.

GDRIVE_ROOT = "/content/gdrive/MyDrive/EK100_MIR"
DATA_DIR    = Path(GDRIVE_ROOT) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR}")

In [ ]:
import subprocess

# Download the JPoSE data archive (~1.64 GB) from Dropbox to Drive if not already present.
# The -c flag resumes a partial download, so it is safe to re-run after an interrupted session.

JPOSE_ZIP_URL = "https://www.dropbox.com/s/bs6y50xkl1rbe20/JPoSE_data.zip?dl=1"
JPOSE_ZIP     = DATA_DIR / "JPoSE_data.zip"

if JPOSE_ZIP.exists():
    print(f"SKIP  JPoSE_data.zip already on Drive ({JPOSE_ZIP.stat().st_size/1e9:.2f} GB)")
else:
    print("Downloading JPoSE_data.zip (~1.64 GB) — this takes a few minutes...")
    r = subprocess.run(
        ["wget", "-q", "--show-progress", "-c", "-O", str(JPOSE_ZIP), JPOSE_ZIP_URL],
        check=True,
    )
    print(f"OK    JPoSE_data.zip ({JPOSE_ZIP.stat().st_size/1e9:.2f} GB)")

In [ ]:
# Download the EK-100 retrieval annotation pickles from the EPIC-KITCHENS GitHub repo.
# These contain the train / test split metadata used by JPoSE at inference time.

ANNOT_BASE = (
    "https://github.com/epic-kitchens/epic-kitchens-100-annotations"
    "/raw/master/retrieval_annotations"
)
ANNOT_DIR = DATA_DIR / "annotations"
ANNOT_DIR.mkdir(exist_ok=True)

for fname in ["EPIC_100_retrieval_train.pkl", "EPIC_100_retrieval_test.pkl"]:
    dest = ANNOT_DIR / fname
    if dest.exists():
        print(f"SKIP  {fname}")
    else:
        print(f"Downloading {fname}...")
        subprocess.run(
            ["wget", "-q", "-O", str(dest), f"{ANNOT_BASE}/{fname}"],
            check=True,
        )
        print(f"OK    {fname} ({dest.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Print a summary of every file stored under DATA_DIR on Drive.
# Confirms all downloads completed and the setup is ready for jpose_base.ipynb.

print(f"\n{'='*50}")
print("  Files on Drive:")
print(f"{'='*50}")
total = 0
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        gb = p.stat().st_size / 1e9
        total += gb
        print(f"  {p.relative_to(DATA_DIR)}  ({gb:.3f} GB)")
print(f"\n  Total: {total:.2f} GB")
print(f"\nDone. You can now run jpose_base.ipynb.")